In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv('../data/clean/cleaned_data.csv')
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [3]:
# MAPPING ORDINAL TO NUMERIC VALUES
ordinal_mappings = {
    'Parental_Involvement': {'Low': 0, 'Medium': 1, 'High': 2},
    'Access_to_Resources': {'Low': 0, 'Medium': 1, 'High': 2},
    'Motivation_Level': {'Low': 0, 'Medium': 1, 'High': 2},
    'Family_Income': {'Low': 0, 'Medium': 1, 'High': 2},
    'Teacher_Quality': {'Low': 0, 'Medium': 1, 'High': 2},
    'Peer_Influence': {'Negative': 0, 'Neutral': 1, 'Positive': 2},
    'Parental_Education_Level': {'High School': 0, 'College': 1, 'Postgraduate': 2},
    'Distance_from_Home': {'Near': 0, 'Moderate': 1, 'Far': 2}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

df_processed = pd.get_dummies(df, drop_first=True)

X = df_processed.drop(['Exam_Score'], axis=1)
y = df_processed['Exam_Score']

In [4]:
# MODEL TRAINING AND EVALUATION
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=98396096)

lr = LinearRegression()
lr.fit(X_train, y_train)

r2 = r2_score(y_test, lr.predict(X_test))
print("=" * 25)
print("PREDICTABILITY")
print("=" * 25)
print(f"Model R² Score: {r2:.4f}\n")

PREDICTABILITY
Model R² Score: 0.7041



In [8]:
# FIT ON THE FULL DATASET TO EXTRACT COEFFICIENTS
lr_full = LinearRegression()
lr_full.fit(X, y)

target_features = ['Hours_Studied', 'Attendance', 'Parental_Involvement']
extracted_coefs = {}

for feature in target_features:
    coef_val = lr_full.coef_[X.columns.get_loc(feature)]
    extracted_coefs[feature] = coef_val

    unit = "hour" if feature == "Hours_Studied" else "%" if feature == "Attendance" else "tier (e.g., Low to Medium)"
    print(f"{feature}: +{coef_val:.2f} points per additional {unit}")

Hours_Studied: +0.29 points per additional hour
Attendance: +0.20 points per additional %
Parental_Involvement: +1.00 points per additional tier (e.g., Low to Medium)


In [ ]:
# SAVE THE RESULTS TO CSV
coef_df = pd.DataFrame(list(extracted_coefs.items()), columns=['Feature', 'Coefficient (Points Added)'])
coef_df = coef_df.sort_values(by='Coefficient (Points Added)', ascending=True)
coef_df.to_csv('q1_prediction_results.csv', index=False)